<img src='sharif_logo.png' alt="SUT logo" width=150 height=150 align=left class="saturate" >

<br>
<font face="Times New Roman">
<div dir=ltr align=center>
<font color=0F5298 size=7>
 Deep Learning <br>
<font color=2565AE size=5>
Computer Engineering Department - Spring 2025  <br>
<font color=3C99D size=5>
          Homework 2:  <br>
<font color=696880 size=4>
           
    

# Assignment Overview

In this assignment, you will explore inference scaling techniques in large language models (LLMs) and evaluate their performance using the Math Benchmark. Throughout the notebook, you will learn about several inference methods, including:

- **Chain-of-Thought (CoT):** A method where the model generates intermediate reasoning steps before providing the final answer.
- **Best-of-n Sampling:** An approach that generates multiple candidate responses and selects the best one based on a scoring function.
- **Beam Search:** A technique that expands several possible sequences simultaneously, choosing the most promising ones based on probability.
- **Self-Refinement:** An iterative process where the model revises its output to improve accuracy and coherence.

The **Math Benchmark** is a suite of challenging mathematical problems designed to test the reasoning and problem-solving capabilities of LLMs. The benchmark includes a variety of questions ranging from basic arithmetic and algebra to more advanced topics such as geometry and calculus. For example, you might be asked to solve an equation like `2x + 5 = 15` or compute the derivative of a function, tasks that assess the model's ability to handle both straightforward and complex mathematical queries.

By the end of this assignment, you will have:
- Gained a deeper understanding of inference time scaling methods in LLMs.
- Compared the effectiveness of different inference techniques using a rigorous math evaluation framework.

Let's dive into the notebook and begin exploring how these methods perform on a challenging set of math problems!


## vLLM: Accelerated Inference Engine for LLMs

vLLM is an open-source project designed to optimize the loading and inference of large language models. By leveraging advanced memory management techniques and dynamic batching, vLLM significantly speeds up the inference process, making it easier to deploy and experiment with LLMs even on hardware with limited resources
So we use vLLM to get results faster.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/MyDrive"

'Colab Notebooks'  'Deep Learning - SUT'


In [ ]:
BASE_ROOT = "/content/drive/MyDrive/Deep Learning - SUT/3 - HW3"
import os

print(os.listdir(BASE_ROOT))

['data', 'Models', 'Results', 'Runs']


In [ ]:
data_root = f"{BASE_ROOT}/data/Q4_Reasoning/"
Models_root = f"{BASE_ROOT}/Models/Q4_Reasoning/"
Results_root = f"{BASE_ROOT}/Results/Q4_Reasoning/"
Runs_root = f"{BASE_ROOT}/Runs/Q4_Reasoning/"

# installing Dependencies

In [ ]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.2/806.2 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 

In [ ]:
import vllm

In [ ]:
!pip install transformers accelerate datasets

In [ ]:
!pip install --upgrade numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 133.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mistral-common 1.11.7 requires numpy<2.4,>=1.25; python_version <= "3.12", but you have numpy 2.5.2 which is incompatible.
numba 0.65.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.2 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but you have cuda-core 1.0.1 which is incompatible.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.3.1 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cuml-cu12 26.2.0 requires numba

In [ ]:
# import os
# os.kill(os.getpid(), 9)

In [ ]:
import transformers, accelerate, datasets


This command launches a vLLM inference server with:
- Model: `DeepSeek-R1-Distill-Qwen-1.5B`
- Port: `8000` (default API endpoint)
- Precision: `half` (FP16) for memory efficiency
- Max context length: `3192` tokens

**Note:**  
🔹 Ensure you're using a GPU runtime (T4 or better) in Colab  
🔹 Only run the next cell if this one executes successfully


In [ ]:
import numpy as np

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.13.0+cu130
PyTorch CUDA: 13.0
CUDA available: True


In [ ]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


In [ ]:
!pip install torchaudio --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached https://download-r2.pytorch.org/whl/cu130/torchaudio-2.11.0%2Bcu130-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached https://download-r2.pytorch.org/whl/cu130/torchaudio-2.11.0%2Bcu130-cp312-cp312-manylinux_2_28_x86_64.whl (1.7 MB)


In [ ]:
import torch
import torchaudio

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("TorchAudio:", torchaudio.__version__)

PyTorch: 2.13.0+cu130
PyTorch CUDA: 13.0
TorchAudio: 2.11.0+cu130


> **Note**:
> instead of downloading the model every single time that we run out of running time, i downloaded it *Once* and i imported it

In [ ]:
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
Models_root

'/content/drive/MyDrive/Deep Learning - SUT/3 - HW3/Models/Q4_Reasoning/'

In [ ]:
model_path = Models_root + "DeepSeek-R1-Distill-Qwen-1.5B"
print(model_path == "/content/drive/MyDrive/Deep Learning - SUT/3 - HW3/Models/Q4_Reasoning/DeepSeek-R1-Distill-Qwen-1.5B")

True


In [ ]:
# from huggingface_hub import snapshot_download

# model_path = model_path

# snapshot_download(
#     repo_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
#     local_dir=model_path
# )

In [ ]:
!ls -lah "{model_path}"

total 3.4G
drwx------ 3 root root 4.0K Aug 17 13:30 .cache
-rw------- 1 root root  679 Aug 17 13:30 config.json
drwx------ 2 root root 4.0K Aug 17 13:30 figures
-rw------- 1 root root  181 Aug 17 13:30 generation_config.json
-rw------- 1 root root 1.5K Aug 17 13:30 .gitattributes
-rw------- 1 root root 1.1K Aug 17 13:30 LICENSE
-rw------- 1 root root 3.4G Aug 17 13:31 model.safetensors
-rw------- 1 root root  16K Aug 17 13:30 README.md
-rw------- 1 root root 3.0K Aug 17 13:30 tokenizer_config.json
-rw------- 1 root root 6.8M Aug 17 13:30 tokenizer.json


* this cell lunches model in background using vllm

In [ ]:
!nohup vllm serve "{model_path}" \
    --port 8000 \
    --dtype half \
    --max-model-len 5192 \
    > vllm.log 2>&1 &

> **Note**:
> ### vLLM Startup Time
>
> After running `vllm serve`, the server may take some time to initialize and load the model.
>
> Therefore, if we immediately run:
>
> ```bash
> !curl http://localhost:8000/v1/models
> ```
>
> we may get:
>
> ```text
> curl: (7) Failed to connect to localhost port 8000
> ```
>
> This does **not necessarily mean that vLLM or the model is broken**. The server may still be initializing or loading the model into GPU memory.
>
> Instead of using a fixed `sleep` time, it is better to wait until the API becomes available:
>
> ```bash
> !sleep 120
> !curl http://localhost:8000/v1/models
> ```
>
> To diagnose startup problems, we can also inspect the vLLM log:
>
> ```bash
> !tail -n 100 vllm.log
> ```
>
> Once the server is fully initialized, the log should contain messages such as:
>
> ```text
> Application startup complete.
> API server: HTTP server started
> ```
>
> **Key takeaway:** `Connection refused` immediately after starting vLLM can simply mean that the server is still starting up. Always check the logs before assuming that the server has failed.

In [ ]:
!sleep 120
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"/content/drive/MyDrive/Deep Learning - SUT/3 - HW3/Models/Q4_Reasoning/DeepSeek-R1-Distill-Qwen-1.5B","object":"model","created":1787145901,"owned_by":"vllm","root":"/content/drive/MyDrive/Deep Learning - SUT/3 - HW3/Models/Q4_Reasoning/DeepSeek-R1-Distill-Qwen-1.5B","parent":null,"max_model_len":5192,"permission":[{"id":"modelperm-847d7e2e2900fc3c","object":"model_permission","created":1787145901,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [ ]:
!nvidia-smi

Wed Aug 19 13:25:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             62W /  400W |   76008MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

1. first of all, downloading the `DeepSeek-R1-Distill-Qwen-1.5B` model from Hugging face, in this step, the model is now on our colab VM, not on our DeepSeek Server.
2. vLLM reads model weights from disk and places them on the GPU for inference:

## LLM Query Function

* This Python function sends prompts to a locally-hosted LLM API and returns the generated response
* you can change max_tokens and temperature as you want



In [ ]:
import requests

def get_llm_response(prompt):
    # Send a prompt to the local vLLM server and return the generated text.
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": model_path,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": 1500,
        "temperature": 0.6
    }

    response = requests.post(
        url,
        json=payload,
        timeout=120
    )

    # Raise an error automatically if the request was unsuccessful.
    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"].strip()

3. making connection through API To send a chat request.
4. With our python command:  
```python:
requests.post(
    "http://localhost:8000/v1/chat/completions",
    json=payload
)
```  
we are actually telling the LLM:
> "Hey LLM running on this machine, take this prompt and answer."

# Test response generation
- testing model with some Math benchmark quesions

In [ ]:
# TODO: Generate a response with these Math benchmark questions

question1 = "How many positive whole-number divisors does 196 have?"
# real answer: 9

question2 = "What is the distance, in units, between the points $(2, -6)$ and $(-4, 3)$? Express your answer in simplest radical form."
# real answer: 3\sqrt{13}

question3 = "Define\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} \\quad q = \\sum_{k = 1}^\\infty \\frac{1}{k^3}.\\]Find a way to write\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3}\\]in terms of $p$ and $q.$"

# real answer: p - q


response = []

for question in [question1, question2, question3]:
    answer = get_llm_response(question)
    response.append(answer)

for i, answer in enumerate(response, start=1):
    print(f"\n{'=' * 20} Question {i} {'=' * 20}")
    print(answer)


==================== Question 1 ====================
Okay, so I need to figure out how many positive whole-number divisors 196 has. Hmm, divisors, right? That means I'm looking for all the whole numbers that can divide 196 without leaving a remainder. For example, 1 and 196 are always divisors because any number is divisible by 1 and itself. But there should be more in between.

I remember that to find the number of divisors, I can use the prime factorization method. Let me try that. So, first, I need to break down 196 into its prime factors. I know that 196 is an even number, so it's divisible by 2. Let me divide it by 2:

196 ÷ 2 = 98.

Okay, so 2 is one prime factor. Now, 98 is also even, so I can divide it by 2 again:

98 ÷ 2 = 49.

Now, 49 is not even, so I can't divide by 2 anymore. Next, I should check if 49 is divisible by the next prime number, which is 3. Let me add the digits of 49: 4 + 9 = 13. Since 13 isn't divisible by 3, 49 isn't either. The next prime number is 5. 49 d

# Math Benchmark Evaluation

This cell is dedicated to evaluating the performance of inference scaling methods on the Math Benchmark dataset. The process works as follows:

- **Dataset Loading:** It loads the MATH-500 dataset, which contains a set of challenging math problems along with their correct solutions.
- **Answer Extraction:** The `extract_answer` function is used to parse and extract the final answer from the generated responses. This function specifically looks for a LaTeX-style format (using `\boxed{...}`) to reliably pinpoint the answer.
- **Normalization and Comparison:** Before comparing, both the predicted answer and the ground truth are normalized using several functions. These functions handle different mathematical expressions, such as fractions, matrices, and algebraic expressions, ensuring that the comparison is fair and accurate regardless of formatting differences.
- **Evaluation Loop:** For each problem:
  - The ground truth answer is extracted from the provided solution.
  - A response is generated by the LLM using a designated function.
  - The predicted answer is then extracted and compared against the ground truth.
  - The results for each problem, including whether the predicted answer is correct, are saved for later analysis.
- **Results Analysis:** After processing all problems, the cell aggregates the results and prints a summary, including the total number of problems evaluated, the number of correct answers, and the overall accuracy.

This evaluation method ensures that the output of each inference technique (such as Chain-of-Thought, Best-of-n, Beam Search, and Self-Refinement) is consistently measured against the Math Benchmark, without altering the original answers or evaluation logic.

**Note:**  

🔹 you don't need to modify this cell. Only rewrite the evaluation function portion then

🔹 you need to run this cell before evaluating.


In [ ]:
import re
import sympy as sp
from typing import Optional

def extract_answer(response: Optional[str]) -> Optional[str]:
    """
    Extract the final answer from an LLM response.

    Priority:
    1. Last \\boxed{...}
    2. Final Answer: ...
    3. Answer: ...
    4. Last display math expression
    5. Last inline math expression
    6. Last math-like line
    """

    # Return None when the model response is empty.
    if response is None:
        return None

    response = str(response).strip()

    if not response:
        return None


    # --------------------------------------------------------
    # 1. Extract the last \\boxed{...} expression
    # --------------------------------------------------------

    boxed_start = response.rfind(r"\boxed{")

    if boxed_start != -1:

        start = boxed_start + len(r"\boxed{")
        brace_count = 1
        pos = start

        while pos < len(response) and brace_count > 0:

            if response[pos] == "{":
                brace_count += 1

            elif response[pos] == "}":
                brace_count -= 1

            pos += 1

        # Return the content if the braces are balanced.
        if brace_count == 0:

            answer = response[start:pos - 1].strip()

            if answer:

                # If the result contains an equality chain,
                # keep only the final expression.
                if "=" in answer:
                    answer = answer.split("=")[-1].strip()

                return answer


    # --------------------------------------------------------
    # 2. Look for "Final Answer: ..."
    # --------------------------------------------------------

    final_patterns = [
        r"Final\s+Answer\s*[:=]\s*(.+)",
        r"FINAL\s+ANSWER\s*[:=]\s*(.+)",
        r"Final\s+answer\s*[:=]\s*(.+)",
    ]

    for pattern in final_patterns:

        matches = re.findall(
            pattern,
            response,
            flags=re.IGNORECASE
        )

        if matches:

            answer = matches[-1].strip()
            answer = answer.splitlines()[0].strip()

            # Keep only the last part of an equality chain.
            if "=" in answer:
                answer = answer.split("=")[-1].strip()

            if answer:
                return answer


    # --------------------------------------------------------
    # 3. Look for "Answer: ..."
    # --------------------------------------------------------

    answer_patterns = [
        r"Answer\s*[:=]\s*(.+)",
        r"ANSWER\s*[:=]\s*(.+)",
        r"The\s+answer\s+is\s*[:=]?\s*(.+)",
    ]

    for pattern in answer_patterns:

        matches = re.findall(
            pattern,
            response,
            flags=re.IGNORECASE
        )

        if matches:

            answer = matches[-1].strip()
            answer = answer.splitlines()[0].strip()

            # Keep only the final part of an equality chain.
            if "=" in answer:
                answer = answer.split("=")[-1].strip()

            if answer:
                return answer


    # --------------------------------------------------------
    # 4. Look for display math at the end
    # --------------------------------------------------------

    display_matches = re.findall(
        r"\\\[(.*?)\\\]",
        response,
        flags=re.DOTALL
    )

    if display_matches:

        answer = display_matches[-1].strip()

        # Example:
        # (p - 1) - (q - 1) = p - 1 - q + 1 = p - q
        # becomes:
        # p - q
        if "=" in answer:
            answer = answer.split("=")[-1].strip()

        if answer:
            return answer


    # --------------------------------------------------------
    # 5. Look for $$ ... $$
    # --------------------------------------------------------

    display_matches = re.findall(
        r"\$\$(.*?)\$\$",
        response,
        flags=re.DOTALL
    )

    if display_matches:

        answer = display_matches[-1].strip()

        if "=" in answer:
            answer = answer.split("=")[-1].strip()

        if answer:
            return answer


    # --------------------------------------------------------
    # 6. Look for inline math at the end
    # --------------------------------------------------------

    inline_matches = re.findall(
        r"\$([^$]+)\$",
        response
    )

    if inline_matches:

        answer = inline_matches[-1].strip()

        if "=" in answer:
            answer = answer.split("=")[-1].strip()

        if answer:
            return answer


    # --------------------------------------------------------
    # 7. Use the last math-like line as a fallback
    # --------------------------------------------------------

    lines = [
        line.strip()
        for line in response.splitlines()
        if line.strip()
    ]

    for line in reversed(lines):

        # Ignore markdown/code delimiters.
        if line in ["```", "```latex", "```text"]:
            continue

        # Ignore long natural-language sentences.
        if len(line) > 120:
            continue

        # If the line contains an equality chain,
        # keep only the final expression.
        if "=" in line:

            candidate = line.split("=")[-1].strip()
            candidate = candidate.rstrip(".,;:")

            if candidate:
                return candidate

        # Accept only lines that look mathematical.
        math_like = re.fullmatch(
            r"[\s\dA-Za-z+\-*/^_=().,{}\\]+",
            line
        )

        if math_like:

            # Reject obvious English sentences.
            words = re.findall(r"[A-Za-z]+", line)

            if len(words) <= 3:

                answer = line.rstrip(".,;:")

                if answer:
                    return answer


    # No reliable final answer was found.
    return None

In [ ]:
questions = [question1, question2, question3]

for i, question in enumerate(questions, 1):

    print("\n" + "#" * 100)
    print(f"QUESTION {i}")
    print("#" * 100)

    response = get_llm_response(question)

    print("\n--- EXTRACTED ANSWER ---")
    print(extract_answer(response))


####################################################################################################
QUESTION 1
####################################################################################################

--- EXTRACTED ANSWER ---
9

####################################################################################################
QUESTION 2
####################################################################################################

--- EXTRACTED ANSWER ---
3\sqrt{13}

####################################################################################################
QUESTION 3
####################################################################################################

--- EXTRACTED ANSWER ---
\frac{2}{27}


In [ ]:
# ============================================================
# Quick validation tests
# ============================================================

tests = [
    ("6+9i", "6 + 9i"),
    ("-2 + 7i", r"\boxed{-2 + 7i}"),
    ("x=5", "5"),
    (r"3\sqrt{13}", r"\boxed{3\sqrt{13}}"),
    ("p-q", "p - q"),
    (r"\left( 3, \frac{\pi}{2} \right)", r"\boxed{(3,\frac{\pi}{2})}"),
    (r"\frac{3}{56}", "3/56"),
    ("9", "9.0"),
    (r"\pi", r"\boxed{\pi}"),
]


for correct, predicted in tests:

    print(
        f"{correct} VS {predicted} => "
        f"{compare_answers(correct, predicted)}"
    )

6+9i VS 6 + 9i => True
-2 + 7i VS \boxed{-2 + 7i} => True
x=5 VS 5 => True
3\sqrt{13} VS \boxed{3\sqrt{13}} => True
p-q VS p - q => True
\left( 3, \frac{\pi}{2} \right) VS \boxed{(3,\frac{\pi}{2})} => False
\frac{3}{56} VS 3/56 => True
9 VS 9.0 => True
\pi VS \boxed{\pi} => True


# Customizable CoT Prompt Template
* modify cot prompt then evaluate on math benchmark


* generate response with cot prompt

In [ ]:
def get_COT_response(problem):
    prompt = COT_PROMPT + "\n" + problem
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }

        ],
    "max_tokens": 1900,
    "temperature": 0.3
    }
    response = requests.post(url, json=payload)
    return response.json()['choices'][0]['message']['content'].strip()

In [ ]:
# final answer should be in this format: (because of extract_answer function you can change it if you want)
#\\[
#\\boxed{your_answer_here}
#\\]

COT_PROMPT = r"""
Solve the following math problem step by step.

After completing your reasoning, give the final answer in exactly this format:

\[
\boxed{{FINAL_ANSWER}}
\]

Replace FINAL_ANSWER with the actual answer.
Do not literally write "FINAL_ANSWER".
Do not stop before providing the boxed final answer.

Problem:
{question}
"""

# Evaluate CoT
* modify response generation part to evalute this method.

In [ ]:
def load_existing_results(filename: str) -> list[Dict]:
    # Load previously saved evaluation results from the JSON file.
    # If the file does not exist yet, return an empty list.
    try:
        with open(filename, "r") as f:
            return json.load(f)

    except FileNotFoundError:
        return []

making sure that the old results are removed.

In [ ]:
import os

results_file = os.path.join(
    Results_root,
    "evaluation_results_math500_deepseek_cot.json"
)

if os.path.exists(results_file):
    os.remove(results_file)
    print("Old evaluation results deleted.")
else:
    print("No previous results found.")

Old evaluation results deleted.


In [ ]:
print(os.path.exists(results_file))

False


In [ ]:
def evaluate_cot():
    os.makedirs("results", exist_ok=True)
    results_file = os.path.join(
        Results_root,
        "evaluation_results_math500_deepseek_cot.json"
    )
    # Load the MATH-500 dataset.
    dataset = load_math500_dataset()

    # Load previously saved results so we can resume the evaluation
    # without re-evaluating problems that have already been processed.
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}

    cnt = 0

    # Iterate through the dataset and evaluate the first 30 problems.
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):

        # Skip problems that were already evaluated in a previous run.
        if idx in processed_indexes:
            continue

        # Stop after evaluating the first 30 problems.
        if idx >= 30:
            break

        # Get the problem statement from the dataset.
        problem_text = item['problem']

        # Extract the reference answer from the official solution.
        correct_answer = extract_answer(item['solution'])

        # Generate a Chain-of-Thought prompt for the current problem.
        prompt = COT_PROMPT.format(question=problem_text)

        # Send the prompt to the locally hosted DeepSeek model
        # through the vLLM OpenAI-compatible API.
        response = get_llm_response(prompt)

        # Extract the final answer from the model's response.
        predicted_answer = extract_answer(response)

        # Compare the model's answer with the reference answer.
        is_correct = compare_answers(
            correct_answer,
            predicted_answer
        )

        # Store the complete evaluation result.
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }

        # Save the result immediately so that the evaluation
        # can be resumed if the Colab session stops unexpectedly.
        save_result(results_file, result)

        # Update the number of correct answers.
        if is_correct:
            cnt += 1

        # Print the current evaluation progress.
        print(f"corrects : {cnt} idx: {idx}")

    # Load all saved results after evaluation is finished.
    final_results = load_existing_results(results_file)

    # Print the final evaluation summary.
    analyze_results(final_results)

In [ ]:
evaluate_cot()

Evaluating problems:   0%|          | 1/500 [00:03<25:03,  3.01s/it]

corrects : 1 idx: 0


Evaluating problems:   0%|          | 2/500 [00:08<39:30,  4.76s/it]

corrects : 2 idx: 1


Evaluating problems:   1%|          | 3/500 [00:12<36:28,  4.40s/it]

...

corrects : 12 idx: 27


Evaluating problems:   6%|▌         | 29/500 [02:22<35:27,  4.52s/it]

corrects : 13 idx: 28


Evaluating problems:   6%|▌         | 30/500 [02:26<38:12,  4.88s/it]

corrects : 14 idx: 29

=== Results Summary ===
Total problems: 30
Correct answers: 14
Accuracy: 46.67%

=== Incorrect Problems ===
Problem 2:
Expected: \frac{14}{3}
Predicted: \dfrac{14}{3}
---
Problem 4:
Expected: \text{Evelyn}
Predicted: De
---
Problem 9:
Expected: 4
Predicted: 121
---
Problem 10:
Expected: 2220
Predicted: None
---
Problem 11:
Expected: \frac{3}{56}
Predicted: -7
---
Problem 12:
Expected: 284
Predicted: 504
---
Problem 14:
Expected: \sqrt{51}
Predicted: sqrt(51). So, DE is sqrt(51).
---
Problem 15:
Expected: 6 - 5i
Predicted: i
---
Problem 17:
Expected: \pi
Predicted: None
---
Problem 18:
Expected: 28
Predicted: None
---
Problem 19:
Expected: 3
Predicted: |a - 1| \geq 2
---
Problem 21:
Expected: 13535
Predicted: 13536
---
Problem 23:
Expected: 5
Predicted: 0
---
Problem 24:
Expected: 10
Predicted: \frac{ -1 \pm 3.2 }{2}
---
Problem 25:
Expected: 1,-2
Predicted: -1
---
Problem 26:
Expected: 144
Predicted: 720
---


## Best-of-N

The Best-of-N approach generates several candidate responses for a problem and then selects the one with the highest average token log-likelihood. This ensures that the final answer, formatted within the `\boxed{}` command, is not only correct in presentation but also statistically the most reliable.


In [ ]:
from collections import defaultdict


SYSTEM_PROMPT = '''You are solving mathematics problems.

Please think step by step.

Important: Always end your solution with the final answer in this format:

\\[
\\boxed{your_answer_here}
\\]

The entire answer should be contained completely within the \\boxed{} command.'''


def best_of_n_response(problem, N=5):

    best_answer = None
    best_avg_likelihood = float('-inf')
    best_responses = []

    prompt = SYSTEM_PROMPT + "\n" + problem

    for t in range(N):

        # --------------------------------------------------------
        # Generate a response with token-level log probabilities
        # --------------------------------------------------------

        url = "http://localhost:8000/v1/chat/completions"

        payload = {
            "model": model_path,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "max_tokens": 1500,
            "temperature": 0.6,

            # Return log probability for generated tokens.
            "logprobs": True
        }

        response = requests.post(
            url,
            json=payload,
            timeout=180
        )

        response.raise_for_status()

        data = response.json()


        # --------------------------------------------------------
        # Iterate over each choice and collect token logprobs
        # --------------------------------------------------------

        for choice in data["choices"]:

            generated_text = choice["message"]["content"].strip()

            token_logprobs = []

            logprob_data = choice.get("logprobs")

            if logprob_data is not None:

                content_logprobs = logprob_data.get("content", [])

                for token_info in content_logprobs:

                    logprob = token_info.get("logprob")

                    if logprob is not None:
                        token_logprobs.append(logprob)


            # ----------------------------------------------------
            # Calculate the average token log-likelihood
            # ----------------------------------------------------

            if token_logprobs:

                avg_log_likelihood = (
                    sum(token_logprobs)
                    / len(token_logprobs)
                )

            else:
                avg_log_likelihood = float("-inf")


            # ----------------------------------------------------
            # Extract the final mathematical answer
            # ----------------------------------------------------

            answer = extract_answer(generated_text)


            # ----------------------------------------------------
            # Store response information
            # ----------------------------------------------------

            best_responses.append({
                "response": generated_text,
                "answer": answer,
                "avg_log_likelihood": avg_log_likelihood
            })


            print(
                f"Sample {t + 1}/{N} | "
                f"Answer: {answer} | "
                f"Avg log-likelihood: {avg_log_likelihood:.4f}"
            )


    # ------------------------------------------------------------
    # Group responses by extracted answer
    # ------------------------------------------------------------

    grouped_responses = defaultdict(list)

    for result in best_responses:

        answer = result["answer"]

        if answer is not None:

            grouped_responses[answer].append(
                result["avg_log_likelihood"]
            )


    # ------------------------------------------------------------
    # Find the best answer based on average likelihood
    # ------------------------------------------------------------

    for answer, likelihoods in grouped_responses.items():

        group_avg_likelihood = (
            sum(likelihoods)
            / len(likelihoods)
        )

        print(
            f"Answer: {answer} | "
            f"Count: {len(likelihoods)} | "
            f"Average likelihood: {group_avg_likelihood:.4f}"
        )

        if group_avg_likelihood > best_avg_likelihood:

            best_avg_likelihood = group_avg_likelihood
            best_answer = answer


    return best_answer

# Evaluate best of n

* modify response generation part to evalute this method.

In [ ]:
def evaluate_best_of_n():

    os.makedirs("results", exist_ok=True)

    results_file = "evaluation_results_math500_deepseek_best_of_n.json"

    dataset = load_math500_dataset()

    existing_results = load_existing_results(results_file)

    processed_indexes = {
        result["index"]
        for result in existing_results
    }

    cnt = 0

    for idx, item in enumerate(
        tqdm(dataset, desc="Evaluating problems")
    ):

        # Skip previously evaluated problems.
        if idx in processed_indexes:
            continue

        # Evaluate only the first 30 problems.
        if idx >= 30:
            break

        problem_text = item["problem"]

        # Extract the reference answer from the official solution.
        correct_answer = extract_answer(
            item["solution"]
        )

        # --------------------------------------------------------
        # Generate the best answer using Best-of-N sampling.
        # --------------------------------------------------------

        response = best_of_n_response(
            problem_text,
            N=5
        )

        # best_of_n_response already returns the final answer.
        predicted_answer = response

        # --------------------------------------------------------
        # Compare the predicted answer with the reference answer.
        # --------------------------------------------------------

        is_correct = compare_answers(
            correct_answer,
            predicted_answer
        )

        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }

        save_result(
            results_file,
            result
        )

        if is_correct:
            cnt += 1

        print(
            f"corrects: {cnt} | "
            f"idx: {idx} | "
            f"expected: {correct_answer} | "
            f"predicted: {predicted_answer}"
        )

    final_results = load_existing_results(
        results_file
    )

    analyze_results(final_results)

In [ ]:
evaluate_best_of_n()

Evaluating problems:   0%|          | 0/500 [00:00<?, ?it/s]

Sample 1/5 | Answer: \left(3, \frac{\pi}{2}\right) | Avg log-likelihood: -0.3124
Sample 2/5 | Answer: (3,\ \frac{\pi}{2}) | Avg log-likelihood: -0.0838
Sample 3/5 | Answer: \left(3, \frac{\pi}{2}\right) | Avg log-likelihood: -0.0806
Sample 4/5 | Answer: (3, \frac{\pi}{2}) | Avg log-likelihood: -0.1010


Evaluating problems:   0%|          | 1/500 [00:11<1:35:40, 11.50s/it]

Sample 5/5 | Answer: \left(3,\frac{\pi}{2}\right) | Avg log-likelihood: -0.1591
Answer: \left(3, \frac{\pi}{2}\right) | Count: 2 | Average likelihood: -0.1965
Answer: (3,\ \frac{\pi}{2}) | Count: 1 | Average likelihood: -0.0838
Answer: (3, \frac{\pi}{2}) | Count: 1 | Average likelihood: -0.1010
Answer: \left(3,\frac{\pi}{2}\right) | Count: 1 | Average likelihood: -0.1591
corrects: 0 | idx: 0 | expected: \left( 3, \frac{\pi}{2} \right) | predicted: (3,\ \frac{\pi}{2})
Sample 1/5 | Answer: ∫₀¹ (ln(1 - t))/t dt | Avg log-likelihood: -0.2533
Sample 2/5 | Answer: p - q | Avg log-likelihood: -0.2682
Sample 3/5 | Answer: -\ln x | Avg log-likelihood: -0.2489
Sample 4/5 | Answer: \frac{1}{8} + \frac{1}{27} + \frac{1}{27} + \frac{1}{64} | Avg log-likelihood: -0.2157


Evaluating problems:   0%|          | 2/500 [00:42<3:12:15, 23.16s/it]

...

Sample 5/5 | Answer: 78 | Avg log-likelihood: -0.1164
Answer: 78 | Count: 5 | Average likelihood: -0.1207
corrects: 10 | idx: 27 | expected: 78 | predicted: 78
Sample 1/5 | Answer: -2 + 7i | Avg log-likelihood: -0.2558
Sample 2/5 | Answer: -2 + 7i | Avg log-likelihood: -0.2847
Sample 3/5 | Answer: x | Avg log-likelihood: -0.3244
Sample 4/5 | Answer: -2 + 7i | Avg log-likelihood: -0.1650


Evaluating problems:   6%|▌         | 29/500 [11:01<3:07:02, 23.83s/it]

Sample 5/5 | Answer: -2 + 7i | Avg log-likelihood: -0.3176
Answer: -2 + 7i | Count: 4 | Average likelihood: -0.2558
Answer: x | Count: 1 | Average likelihood: -0.3244
corrects: 11 | idx: 28 | expected: -2 + 7i | predicted: -2 + 7i
Sample 1/5 | Answer: 225 | Avg log-likelihood: -0.1161
Sample 2/5 | Answer: 225 | Avg log-likelihood: -0.1192
Sample 3/5 | Answer: 225 | Avg log-likelihood: -0.1317
Sample 4/5 | Answer: 225 | Avg log-likelihood: -0.0977


Evaluating problems:   6%|▌         | 30/500 [11:11<2:55:13, 22.37s/it]

Sample 5/5 | Answer: 225 | Avg log-likelihood: -0.0968
Answer: 225 | Count: 5 | Average likelihood: -0.1123
corrects: 12 | idx: 29 | expected: 225 | predicted: 225

=== Results Summary ===
Total problems: 30
Correct answers: 12
Accuracy: 40.00%

=== Incorrect Problems ===
Problem 0:
Expected: \left( 3, \frac{\pi}{2} \right)
Predicted: (3,\ \frac{\pi}{2})
---
Problem 1:
Expected: p - q
Predicted: \frac{1}{8} + \frac{1}{27} + \frac{1}{27} + \frac{1}{64}
---
Problem 2:
Expected: \frac{14}{3}
Predicted: \dfrac{14}{3}
---
Problem 4:
Expected: \text{Evelyn}
Predicted: 3.6
---
Problem 6:
Expected: 27
Predicted: Therefore
---
Problem 7:
Expected: 90^\circ
Predicted: 90
---
Problem 9:
Expected: 4
Predicted: Then, 24 *
---
Problem 10:
Expected: 2220
Predicted: 222
---
Problem 11:
Expected: \frac{3}{56}
Predicted: 1 \) is also a root of \( x^2 - 1 \
---
Problem 12:
Expected: 284
Predicted: 280
---
Problem 15:
Expected: 6 - 5i
Predicted: (a + bi) \cdot \left( \frac{a}{2} + i\frac{a}{2} \right )
--

## Beam Search

This cell implements a beam search strategy for generating candidate reasoning chains. The method generates multiple continuations at each reasoning step, scoring each candidate based on its average token log-likelihood. By retaining and expanding only the top candidates, the approach efficiently searches for the most promising chain-of-thought that leads to the final answer in the required format.


In [63]:
def call_qwen_model_raw(prompt, step_num, temperature=0.8):
    """
    Sends a request to the local Qwen endpoint and returns the generated text
    along with the average token log-probability.
    """

    url = "http://localhost:8000/v1/chat/completions"

    # Use fewer tokens for intermediate reasoning steps
    # and more tokens for later/final steps.
    if step_num == 1:
        max_tokens = 300
    elif step_num == 2:
        max_tokens = 400
    else:
        max_tokens = 700

    payload = {
        "model": model_path,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
        "logprobs": True
    }

    # --------------------------------------------------------
    # Send a request to the local Qwen/DeepSeek model.
    # --------------------------------------------------------

    response = requests.post(
        url,
        json=payload,
        timeout=180
    )

    response.raise_for_status()

    data = response.json()

    choice = data["choices"][0]

    output_text = choice["message"]["content"].strip()


    # --------------------------------------------------------
    # Iterate over generated tokens and collect logprobs.
    # --------------------------------------------------------

    token_logprobs = []

    logprob_data = choice.get("logprobs")

    if logprob_data is not None:

        for token_info in logprob_data.get("content", []):

            logprob = token_info.get("logprob")

            if logprob is not None:
                token_logprobs.append(logprob)


    # --------------------------------------------------------
    # Calculate the average token log-likelihood.
    # --------------------------------------------------------

    if token_logprobs:
        avg_token_prob = (
            sum(token_logprobs)
            / len(token_logprobs)
        )
    else:
        avg_token_prob = float("-inf")


    return output_text, avg_token_prob, len(token_logprobs)



class BeamCandidate:
    def __init__(
        self,
        sequence,
        cumulative_log_prob,
        step_scores,
        finished=False,
        num_token=0
    ):
        self.sequence = sequence
        self.cumulative_log_prob = cumulative_log_prob
        self.step_scores = step_scores
        self.finished = finished
        self.num_token = num_token

    def __repr__(self):
        return (
            f"BeamCandidate("
            f"score={self.cumulative_log_prob:.3f}, "
            f"finished={self.finished}, "
            f"sequence={self.sequence})"
        )



def generate_reasoning_steps(context, step_num, top_k):
    """
    For a given candidate reasoning chain (context), generate top_k candidate continuations
    for the current reasoning step. Each candidate is scored using the average
    token log-probability as a proxy for quality.
    """

    candidates = []


    # --------------------------------------------------------
    # Generate multiple candidate continuations.
    # --------------------------------------------------------

    for i in range(top_k):

        # ----------------------------------------------------
        # Step 1: Understand the problem and propose a strategy.
        # ----------------------------------------------------

        if step_num == 1:

            candidate_prompt = (
                context
                + """

Continue solving the problem.

For this step:
- Identify the important mathematical information.
- Determine the variables and constraints.
- Choose a suitable strategy or formula.
- Do not give the final answer yet.
- Write only the next useful reasoning step.
"""
            )


        # ----------------------------------------------------
        # Step 2: Perform the main derivation/calculation.
        # ----------------------------------------------------

        elif step_num == 2:

            candidate_prompt = (
                context
                + """

Continue from the reasoning above.

For this step:
- Carry out the main mathematical derivation or calculation.
- Check the intermediate calculations carefully.
- Correct any obvious mistake if necessary.
- Do not give the final answer yet unless it is unavoidable.
- Write only the next useful reasoning step.
"""
            )


        # ----------------------------------------------------
        # Final step: Finish the solution and give boxed answer.
        # ----------------------------------------------------

        else:

            candidate_prompt = (
                context
                + """

Continue from the reasoning above and complete the solution.

For this step:
- Finish the remaining calculations.
- Verify the reasoning.
- Give the final answer at the end in exactly this format:

\\[
\\boxed{FINAL_ANSWER}
\\]

Replace FINAL_ANSWER with the actual mathematical answer.
"""
            )


        # ----------------------------------------------------
        # Call the model and obtain candidate score.
        # ----------------------------------------------------

        candidate_step, avg_token_prob, num_token = call_qwen_model_raw(
            candidate_prompt,
            step_num
        )


        # ----------------------------------------------------
        # A candidate is finished only if it contains a boxed
        # final answer.
        # ----------------------------------------------------

        finished = r"\boxed{" in candidate_step


        candidates.append(
            (
                candidate_step,
                avg_token_prob,
                num_token,
                finished
            )
        )


    return candidates



def beam_search(
    init_problem_prompt,
    beam_width=3,
    max_steps=3,
    top_k=2
):
    """
    Implements a beam search over reasoning steps.
    """

    # --------------------------------------------------------
    # Build the initial prompt.
    # --------------------------------------------------------

    prompt = (
        SYSTEM_PROMPT
        + "\n\nProblem:\n"
        + init_problem_prompt
    )


    # --------------------------------------------------------
    # Create the initial beam candidate.
    # --------------------------------------------------------

    initial_candidate = BeamCandidate(
        sequence=prompt,
        cumulative_log_prob=0.0,
        step_scores=[],
        finished=False,
        num_token=0
    )


    beams = [initial_candidate]


    # --------------------------------------------------------
    # Expand reasoning chains step by step.
    # --------------------------------------------------------

    for step_num in range(1, max_steps + 1):

        new_beams = []


        for candidate in beams:

            # ------------------------------------------------
            # Propagate finished candidates unchanged.
            # ------------------------------------------------

            if candidate.finished:

                new_beams.append(candidate)

                continue


            step_candidates = generate_reasoning_steps(
                candidate.sequence,
                step_num,
                top_k
            )


            for (
                step_text,
                score,
                num_token,
                finished
            ) in step_candidates:


                # ------------------------------------------------
                # Append the new reasoning step.
                # ------------------------------------------------

                new_sequence = (
                    candidate.sequence
                    + "\n\n"
                    + step_text
                )


                # ------------------------------------------------
                # Compute average logprob across ALL tokens
                # generated so far.
                #
                # Weighted average is required because different
                # reasoning steps may have different lengths.
                # ------------------------------------------------

                old_num_token = candidate.num_token

                total_num_token = (
                    old_num_token
                    + num_token
                )


                if total_num_token > 0:

                    cumulative_log_prob = (
                        candidate.cumulative_log_prob
                        * old_num_token
                        +
                        score
                        * num_token
                    ) / total_num_token

                else:

                    cumulative_log_prob = float("-inf")


                # ------------------------------------------------
                # Create a new expanded beam candidate.
                # ------------------------------------------------

                new_candidate = BeamCandidate(
                    sequence=new_sequence,
                    cumulative_log_prob=cumulative_log_prob,
                    step_scores=(
                        candidate.step_scores
                        + [score]
                    ),
                    finished=finished,
                    num_token=total_num_token
                )


                new_beams.append(new_candidate)


        # --------------------------------------------------------
        # Stop if no candidates were generated.
        # --------------------------------------------------------

        if not new_beams:
            break


        # --------------------------------------------------------
        # Sort candidates by cumulative average logprob
        # and keep only the top beam_width candidates.
        # --------------------------------------------------------

        new_beams.sort(
            key=lambda beam: beam.cumulative_log_prob,
            reverse=True
        )

        beams = new_beams[:beam_width]


        # --------------------------------------------------------
        # Stop early if all surviving beams are finished.
        # --------------------------------------------------------

        if all(beam.finished for beam in beams):
            break


    # ------------------------------------------------------------
    # Select only finished beams.
    # ------------------------------------------------------------

    finished_beams = [
        beam
        for beam in beams
        if beam.finished
    ]


    # ------------------------------------------------------------
    # Choose the highest-scoring finished beam.
    # If no beam finished, use the best unfinished candidate.
    # ------------------------------------------------------------

    if finished_beams:

        best_candidate = max(
            finished_beams,
            key=lambda beam: beam.cumulative_log_prob
        )

    else:

        best_candidate = max(
            beams,
            key=lambda beam: beam.cumulative_log_prob
        )


    return best_candidate



def run_qwen_beam_search(
    problem,
    beam_width,
    max_steps,
    top_k,
    log_level
):
    """
    Sets up the sample prompt, performs beam search,
    and extracts the final answer.
    """

    # --------------------------------------------------------
    # Run beam search for the given math problem.
    # --------------------------------------------------------

    best_candidate = beam_search(
        init_problem_prompt=problem,
        beam_width=beam_width,
        max_steps=max_steps,
        top_k=top_k
    )


    # --------------------------------------------------------
    # Optionally display beam-search information.
    # --------------------------------------------------------

    if log_level > 0:

        print("\n===== BEST BEAM =====")
        print(
            "Cumulative score:",
            best_candidate.cumulative_log_prob
        )
        print(
            "Number of tokens:",
            best_candidate.num_token
        )
        print(
            "Step scores:",
            best_candidate.step_scores
        )


    if log_level > 1:

        print("\n===== FULL REASONING CHAIN =====")
        print(best_candidate.sequence)


    # --------------------------------------------------------
    # Extract the final mathematical answer.
    # --------------------------------------------------------

    final_answer = extract_answer(
        best_candidate.sequence
    )


    return final_answer

# Evaluate beam search
* modify response generation part to evalute this method.

In [64]:
def evaluate_beam_search():

    os.makedirs("results", exist_ok=True)

    results_file = "evaluation_results_math500_deepseek_beam_search.json"

    dataset = load_math500_dataset()

    existing_results = load_existing_results(results_file)

    processed_indexes = {
        result["index"]
        for result in existing_results
    }

    # Count previously correct answers when resuming.
    cnt = sum(
        result["is_correct"]
        for result in existing_results
    )

    for idx, item in enumerate(
        tqdm(dataset, desc="Evaluating problems")
    ):

        # Skip problems that were already evaluated.
        if idx in processed_indexes:
            continue

        # Evaluate only the first 30 problems.
        if idx >= 30:
            break

        problem_text = item["problem"]

        # Extract the reference answer from the official solution.
        correct_answer = extract_answer(
            item["solution"]
        )

        # --------------------------------------------------------
        # Generate a response using beam search.
        # --------------------------------------------------------

        best_candidate = beam_search(
            init_problem_prompt=problem_text,
            beam_width=3,
            max_steps=3,
            top_k=2
        )

        # Store the complete reasoning chain.
        response = best_candidate.sequence

        # Extract the final answer from the best beam.
        predicted_answer = extract_answer(response)

        # --------------------------------------------------------
        # Compare the predicted answer with the reference answer.
        # --------------------------------------------------------

        is_correct = compare_answers(
            correct_answer,
            predicted_answer
        )

        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }

        save_result(
            results_file,
            result
        )

        if is_correct:
            cnt += 1

        print(
            f"corrects: {cnt} | "
            f"idx: {idx} | "
            f"expected: {correct_answer} | "
            f"predicted: {predicted_answer}"
        )

    final_results = load_existing_results(
        results_file
    )

    analyze_results(final_results)

In [65]:
evaluate_beam_search()

Evaluating problems:   0%|          | 1/500 [00:12<1:44:18, 12.54s/it]

corrects: 0 | idx: 0 | expected: \left( 3, \frac{\pi}{2} \right) | predicted: **


Evaluating problems:   0%|          | 2/500 [00:39<2:53:58, 20.96s/it]

corrects: 0 | idx: 1 | expected: p - q | predicted: 2}^\infty \frac{1}{n^3}


Evaluating problems:   1%|          | 3/500 [01:01<2:57:45, 21.46s/it]

...

corrects: 9 | idx: 27 | expected: 78 | predicted: 78


Evaluating problems:   6%|▌         | 29/500 [10:02<2:16:59, 17.45s/it]

corrects: 10 | idx: 28 | expected: -2 + 7i | predicted: -2 + 7i


Evaluating problems:   6%|▌         | 30/500 [10:26<2:43:36, 20.89s/it]

corrects: 11 | idx: 29 | expected: 225 | predicted: 225

=== Results Summary ===
Total problems: 30
Correct answers: 11
Accuracy: 36.67%

=== Incorrect Problems ===
Problem 0:
Expected: \left( 3, \frac{\pi}{2} \right)
Predicted: **
---
Problem 1:
Expected: p - q
Predicted: 2}^\infty \frac{1}{n^3}
---
Problem 2:
Expected: \frac{14}{3}
Predicted: \dfrac{14}{3}
---
Problem 4:
Expected: \text{Evelyn}
Predicted: 3.6
---
Problem 7:
Expected: 90^\circ
Predicted: -4z
---
Problem 9:
Expected: 4
Predicted: 5
---
Problem 10:
Expected: 2220
Predicted: 222
---
Problem 11:
Expected: \frac{3}{56}
Predicted: \frac{n}{n^2 - 1}
---
Problem 12:
Expected: 284
Predicted: 260
---
Problem 14:
Expected: \sqrt{51}
Predicted: 8.54
---
Problem 15:
Expected: 6 - 5i
Predicted: \sqrt{2} - 3\sqrt{2}i
---
Problem 17:
Expected: \pi
Predicted: \boxed{your_answer_here}
---
Problem 18:
Expected: 28
Predicted: 56^\circ
---
Problem 19:
Expected: 3
Predicted: a^4 - 8a^3 + 18a^2 - 27 \geq 0
---
Problem 21:
Expected: 13535
Pr

## Self-Refinement

This approach begins by generating an initial solution using the given prompt. It then iteratively refines this output by providing the model with targeted feedback and asking it to improve its response. The process continues until the feedback indicates that no further refinement is necessary, ensuring that the final answer—properly formatted within the `\boxed{}` command—is as accurate and well-reasoned as possible.


In [66]:
SYSTEM_PROMPT = '''You are solving mathematics problems.

Please think step by step.

Important: Always end your solution with the final answer in this format:

\\[
\\boxed{your_answer_here}
\\]

The entire answer should be contained completely within the \\boxed{} command.'''


def generate_content(prompt):
    """
    Send a prompt to the local vLLM endpoint and return the generated text.
    """

    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": model_path,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": 1500,
        "temperature": 0.6
    }

    # Send the request to the locally hosted model.
    response = requests.post(
        url,
        json=payload,
        timeout=180
    )

    # Raise an exception if the request fails.
    response.raise_for_status()

    data = response.json()

    # Extract the generated assistant response.
    output_text = data["choices"][0]["message"]["content"].strip()

    return output_text


def self_refine(problem, max_iter=2):
    """
    Solve a math problem, critique the solution, and refine it
    iteratively when necessary.
    """

    prompt = SYSTEM_PROMPT + "\n\nProblem:\n" + problem

    # --------------------------------------------------------
    # Generate the initial solution.
    # --------------------------------------------------------

    current_output = generate_content(prompt)


    for iteration in range(max_iter):

        # ----------------------------------------------------
        # Ask the model to critique the current solution.
        #
        # The response format is intentionally structured so
        # that we can easily determine whether refinement is
        # required.
        # ----------------------------------------------------

        feedback_prompt = f"""
You are reviewing a mathematical solution.

Original problem and instructions:

{prompt}

Current solution:

{current_output}

Carefully check the solution for:
- mathematical errors
- logical errors
- incorrect calculations
- missing reasoning
- an incorrect final answer

Respond in exactly this format:

NEEDS_REFINEMENT: YES or NO
FEEDBACK: your feedback here

If the solution is fully correct, write:

NEEDS_REFINEMENT: NO
FEEDBACK: The solution is correct.
"""


        # ----------------------------------------------------
        # Generate feedback for the current solution.
        # ----------------------------------------------------

        feedback_response = generate_content(
            feedback_prompt
        )


        # ----------------------------------------------------
        # Parse whether refinement is required.
        # ----------------------------------------------------

        needs_refinement = False

        for line in feedback_response.splitlines():

            if line.strip().upper().startswith(
                "NEEDS_REFINEMENT:"
            ):

                value = line.split(
                    ":",
                    1
                )[1].strip().upper()

                needs_refinement = (
                    value == "YES"
                )

                break


        # ----------------------------------------------------
        # Stop early if the model considers the solution correct.
        # ----------------------------------------------------

        if not needs_refinement:
            break


        # ----------------------------------------------------
        # Create a refinement prompt using the original
        # problem, current solution, and reviewer feedback.
        # ----------------------------------------------------

        refine_prompt = f"""
You are solving the following mathematics problem.

Original problem and instructions:

{prompt}

Previous solution:

{current_output}

Reviewer feedback:

{feedback_response}

Rewrite the solution by correcting all identified problems.

Think carefully and verify the calculations.

Important: End the corrected solution with the final answer
in exactly this format:

\\[
\\boxed{{your_answer_here}}
\\]

Return the complete corrected solution.
"""


        # ----------------------------------------------------
        # Generate the refined solution.
        # ----------------------------------------------------

        refined_output = generate_content(
            refine_prompt
        )


        # Update the current solution for the next iteration.
        current_output = refined_output


    # --------------------------------------------------------
    # Extract the final mathematical answer.
    # --------------------------------------------------------

    answer = extract_answer(
        current_output
    )

    return answer

# Evaluate Self-Refinement
* modify response generation part to evalute this method.

In [67]:
def evaluate_self_refiner():

    os.makedirs("results", exist_ok=True)

    results_file = "evaluation_results_math500_deepseek_self_refiner.json"

    dataset = load_math500_dataset()

    existing_results = load_existing_results(results_file)

    processed_indexes = {
        result["index"]
        for result in existing_results
    }

    # Count already-correct answers when resuming.
    cnt = sum(
        result["is_correct"]
        for result in existing_results
    )

    for idx, item in enumerate(
        tqdm(dataset, desc="Evaluating problems")
    ):

        # Skip previously evaluated problems.
        if idx in processed_indexes:
            continue

        # Evaluate only the first 30 problems.
        if idx >= 30:
            break

        problem_text = item["problem"]

        # Extract the reference answer from the official solution.
        correct_answer = extract_answer(
            item["solution"]
        )

        ##########################################################

        # Generate the final answer using self-refinement.
        response = self_refine(
            problem_text,
            max_iter=2
        )

        # self_refine already returns the extracted final answer.
        predicted_answer = response

        ##########################################################

        # Compare the predicted answer with the reference answer.
        is_correct = compare_answers(
            correct_answer,
            predicted_answer
        )

        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }

        save_result(
            results_file,
            result
        )

        if is_correct:
            cnt += 1

        print(
            f"corrects: {cnt} | "
            f"idx: {idx} | "
            f"expected: {correct_answer} | "
            f"predicted: {predicted_answer}"
        )

    final_results = load_existing_results(
        results_file
    )

    analyze_results(final_results)

In [68]:
evaluate_self_refiner()

Evaluating problems:   0%|          | 1/500 [00:03<26:34,  3.20s/it]

corrects: 0 | idx: 0 | expected: \left( 3, \frac{\pi}{2} \right) | predicted: \left(3,\ \frac{\pi}{2}\right)


Evaluating problems:   0%|          | 2/500 [00:15<1:09:25,  8.36s/it]

corrects: 0 | idx: 1 | expected: p - q | predicted: k+1}^\infty \frac{1}{m^3}


Evaluating problems:   1%|          | 3/500 [00:20<56:16,  6.79s/it]  

...

corrects: 10 | idx: 27 | expected: 78 | predicted: 78


Evaluating problems:   6%|▌         | 29/500 [03:53<1:12:40,  9.26s/it]

corrects: 10 | idx: 28 | expected: -2 + 7i | predicted: [-2; 7]. So, again, that gives us the complex number -2 + 7i


Evaluating problems:   6%|▌         | 30/500 [03:57<1:02:06,  7.93s/it]

corrects: 11 | idx: 29 | expected: 225 | predicted: 225

=== Results Summary ===
Total problems: 30
Correct answers: 11
Accuracy: 36.67%

=== Incorrect Problems ===
Problem 0:
Expected: \left( 3, \frac{\pi}{2} \right)
Predicted: \left(3,\ \frac{\pi}{2}\right)
---
Problem 1:
Expected: p - q
Predicted: k+1}^\infty \frac{1}{m^3}
---
Problem 2:
Expected: \frac{14}{3}
Predicted: \dfrac{14}{3}
---
Problem 6:
Expected: 27
Predicted: **Final Answer**
---
Problem 7:
Expected: 90^\circ
Predicted: 90
---
Problem 9:
Expected: 4
Predicted: 121
---
Problem 11:
Expected: \frac{3}{56}
Predicted: So, (i -
---
Problem 12:
Expected: 284
Predicted: 286
---
Problem 14:
Expected: \sqrt{51}
Predicted: 7
---
Problem 15:
Expected: 6 - 5i
Predicted: First, \( \sqrt{2}
---
Problem 17:
Expected: \pi
Predicted: a \sin\left(b\left(x + \frac{c}{b}\right)\right) + d \)
---
Problem 18:
Expected: 28
Predicted: None
---
Problem 19:
Expected: 3
Predicted: (a - 1)^2 - 4 \
---
Problem 21:
Expected: 13535
Predicted: s_6
---